# Lung Disease Management — Training Pipeline v2
## Step-by-step instructions

**Before starting:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run each cell ONE BY ONE in order
3. Wait for each cell to finish before running the next

**Expected total time:** ~3 hours on T4 GPU

## ✅ Step 1: Check GPU
Run this first. You must see 'CUDA: True' before proceeding.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('ERROR: No GPU! Go to Runtime -> Change runtime type -> T4 GPU and restart')

## ✅ Step 2: Clone project from GitHub

In [ ]:
import os, sys, shutil

os.chdir('/content')
if os.path.exists('/content/LDM'):
    shutil.rmtree('/content/LDM')

!git clone https://github.com/Sarnika-anbu/Lung_Disease_Management.git /content/LDM

os.chdir('/content/LDM')
sys.path.insert(0, '/content/LDM')

assert os.path.exists('src/training/train.py'), 'ERROR: Clone failed!'
print('✅ Project cloned. Working dir:', os.getcwd())
!ls

## ✅ Step 3: Install packages
This takes 2-3 minutes.

In [ ]:
import os
os.chdir('/content/LDM')

!pip install -q kaggle
!pip install -q timm==0.9.16
!pip install -q grad-cam==1.5.0
!pip install -q librosa==0.10.2
!pip install -q noisereduce==3.0.2
!pip install -q reportlab==4.0.9
!pip install -q pypdf==4.2.0
!pip install -q python-multipart
print('✅ All packages installed')

## ✅ Step 4: Set Kaggle credentials
**Edit the two lines below with your Kaggle username and API key.**

Get your key from: https://www.kaggle.com/settings → Create New Token

In [ ]:
import os, json
os.chdir('/content/LDM')

KAGGLE_USERNAME = 'YOUR_USERNAME_HERE'   # <-- EDIT THIS
KAGGLE_KEY      = 'YOUR_API_KEY_HERE'    # <-- EDIT THIS

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Test credentials
result = os.system('kaggle datasets list --max-size 1 > /dev/null 2>&1')
if result == 0:
    print('✅ Kaggle credentials valid')
else:
    print('❌ Kaggle credentials invalid - check your username and API key')

## ✅ Step 5: Download ICBHI dataset (~1.2 GB, takes 5-10 min)

In [ ]:
import os
os.chdir('/content/LDM')
os.makedirs('data/raw/icbhi', exist_ok=True)

print('Downloading ICBHI 2017 dataset...')
!kaggle datasets download -d vbookshelf/respiratory-sound-database -p data/raw/icbhi/ --unzip

wav_count = len(list(__import__('pathlib').Path('data/raw/icbhi').rglob('*.wav')))
print(f'\n✅ Downloaded {wav_count} WAV files')
assert wav_count >= 900, f'ERROR: Expected ~920 files, got {wav_count}'

## ✅ Step 6: Preprocess audio (30-90 min)
Converts WAV files to mel spectrograms. Watch for 'Processed 920 files'.

In [ ]:
import os
os.chdir('/content/LDM')
!PYTHONPATH=/content/LDM python src/data/preprocess.py

## ✅ Step 7: Build balanced dataset splits
This fixes the class imbalance by oversampling minority classes to 150 each.

In [ ]:
import os, sys, re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

os.chdir('/content/LDM')
sys.path.insert(0, '/content/LDM')

# Official ICBHI patient-diagnosis mapping
DIAGNOSIS = {
    '101':'URTI','102':'Healthy','103':'COPD','104':'COPD','105':'Healthy',
    '106':'COPD','107':'COPD','108':'Healthy','109':'COPD','110':'URTI',
    '111':'Asthma','112':'COPD','113':'COPD','114':'Healthy','115':'COPD',
    '116':'Healthy','117':'Healthy','118':'Bronchiectasis','119':'Bronchiectasis',
    '120':'COPD','121':'Healthy','122':'Healthy','123':'Healthy','124':'Healthy',
    '125':'COPD','126':'COPD','127':'Healthy','128':'Healthy','129':'COPD',
    '130':'Healthy','131':'COPD','132':'COPD','133':'Healthy','134':'COPD',
    '135':'URTI','136':'COPD','137':'COPD','138':'Bronchiectasis','139':'COPD',
    '140':'Healthy','141':'COPD','142':'Healthy','143':'Healthy','144':'COPD',
    '145':'Healthy','146':'COPD','147':'Healthy','148':'COPD','149':'Healthy',
    '150':'COPD','151':'Healthy','152':'Healthy','153':'Healthy','154':'Healthy',
    '155':'Healthy','156':'COPD','157':'Healthy','158':'Healthy','159':'Healthy',
    '160':'Pneumonia','161':'Pneumonia','162':'Pneumonia','163':'Pneumonia',
    '164':'Healthy','165':'URTI','166':'Healthy','167':'COPD','168':'Healthy',
    '169':'Healthy','170':'Healthy','171':'Healthy','172':'Healthy','173':'COPD',
    '174':'COPD','175':'Healthy','176':'Asthma','177':'COPD','178':'Healthy',
    '179':'Healthy','180':'Bronchiectasis','181':'COPD','182':'Healthy',
    '183':'Healthy','184':'Healthy','185':'Bronchiectasis','186':'Healthy',
    '187':'COPD','188':'Healthy','189':'Healthy','190':'COPD','191':'Healthy',
    '192':'COPD','193':'COPD','194':'Healthy','195':'COPD','196':'Healthy',
    '197':'COPD','198':'Healthy','199':'COPD','200':'COPD','201':'Healthy',
    '202':'Healthy','203':'Healthy','204':'Healthy','205':'Healthy',
    '206':'Bronchiolitis','207':'Bronchiolitis','208':'Bronchiolitis',
    '209':'Bronchiolitis','210':'Healthy','211':'COPD','212':'COPD',
    '213':'COPD','214':'Healthy','215':'URTI','216':'Healthy','217':'Asthma',
    '218':'Healthy','219':'COPD','220':'Healthy','221':'Asthma','222':'Healthy',
    '223':'Healthy','224':'Healthy','225':'COPD','226':'Healthy',
}

# Collect all preprocessed spectrograms
spec_dir = 'data/processed/spectrograms'
rows = []
if os.path.exists(spec_dir):
    for npy_file in sorted(os.listdir(spec_dir)):
        if not npy_file.endswith('.npy'):
            continue
        stem = npy_file[:-4]
        match = re.match(r'^(\d+)_', stem)
        if match:
            pid = match.group(1)
            diagnosis = DIAGNOSIS.get(pid)
            if diagnosis:
                rows.append({
                    'patient_id': pid,
                    'recording_id': stem,
                    'segment_id': 0,
                    'spectrogram_path': os.path.abspath(os.path.join(spec_dir, npy_file)),
                    'disease_class': diagnosis,
                })

df = pd.DataFrame(rows)
print(f'Raw samples: {len(df)}')
print('Class distribution:')
print(df['disease_class'].value_counts())

assert len(df) > 0, 'ERROR: No spectrograms found. Rerun Step 6.'

# Balance: oversample each class to TARGET_COUNT
TARGET = 150
balanced = []
for cls in df['disease_class'].unique():
    cls_df = df[df['disease_class'] == cls]
    if len(cls_df) < TARGET:
        balanced.append(resample(cls_df, n_samples=TARGET, replace=True, random_state=42))
    else:
        balanced.append(cls_df.sample(n=TARGET, random_state=42))

balanced_df = pd.concat(balanced, ignore_index=True)
print(f'\nBalanced: {len(balanced_df)} samples ({TARGET} per class)')
print(balanced_df['disease_class'].value_counts())

# Stratified 80/20 split
train_df, test_df = train_test_split(
    balanced_df, test_size=0.2,
    stratify=balanced_df['disease_class'], random_state=42
)

os.makedirs('data/processed/splits', exist_ok=True)
train_df.to_csv('data/processed/splits/train.csv', index=False)
test_df.to_csv('data/processed/splits/test.csv', index=False)

print(f'\n✅ Splits saved: Train={len(train_df)}, Test={len(test_df)}')
print('Train classes:', train_df['disease_class'].value_counts().to_dict())

## ✅ Step 8: Train model (~2-3 hours on T4)
Watch for epoch numbers increasing. Training runs 50 epochs + 15 fine-tuning epochs.

**Keep the Colab tab open and active to prevent disconnection.**

In [ ]:
import os
os.chdir('/content/LDM')

# Remove old checkpoint
if os.path.exists('checkpoints/best.pth'):
    os.remove('checkpoints/best.pth')
    print('Removed old checkpoint')

print('Starting training...')
print('Expected: 50 stage-1 epochs + 15 stage-2 epochs')
print('Watch for ICBHI score improving above 0.70\n')
!PYTHONPATH=/content/LDM python src/training/train.py

## ✅ Step 9: Evaluate model

In [ ]:
import os
os.chdir('/content/LDM')
!PYTHONPATH=/content/LDM python src/training/evaluate.py

# Display plots
from IPython.display import Image, display
if os.path.exists('outputs/confusion_matrix.png'):
    display(Image('outputs/confusion_matrix.png'))
if os.path.exists('outputs/reliability_diagram.png'):
    display(Image('outputs/reliability_diagram.png'))

## ✅ Step 10: Download checkpoint to your PC
Save this file — you need it to run the app locally.

In [ ]:
import os, torch
os.chdir('/content/LDM')

if os.path.exists('checkpoints/best.pth'):
    ckpt = torch.load('checkpoints/best.pth', map_location='cpu', weights_only=False)
    print(f'Best ICBHI score: {ckpt["score"]:.4f}')
    print(f'Trained for {ckpt["epoch"]} epochs')
    
    from google.colab import files
    files.download('checkpoints/best.pth')
    print('\n✅ Download started - save best.pth to:')
    print('   C:\\Users\\sarni\\OneDrive\\Desktop\\Lung_Disease_Management\\checkpoints\\')
else:
    print('ERROR: No checkpoint found. Run Step 8 first.')